In [1]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Pixiedust database opened successfully


Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


In [21]:
Cohort_Lab_Original_allcols = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/cohort_lab_Paired_additionalCols_Feb1924.parquet")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
Cohort_Lab_Original_allcols.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- loincclass: string (nullable = true)
 |-- interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)
 |-- value: string (nullable = true)
 |-- modifier: string (nullable = true)
 |-- refLowtype: string (nullable = true)
 |-- refLowtextvalue: string (nullable = true)
 |-- refLowRange: string (nullable = true)
 |-- refHightype: string (nullable = true)
 |-- refHightextvalue: string (nullable = true)
 |-- refHighRange: string (nullable = true)



In [28]:
from pyspark.sql.functions import col
# Filter records with personid count > 1 and < 10
filtered_df = Cohort_Lab_Original_allcols \
    .groupBy("personid") \
    .count() \
    .filter((col("count") > 1) & (col("count") < 10)) \
    .join(Cohort_Lab_Original_allcols, "personid", "inner")

# Show the filtered DataFrame
filtered_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-----+-------+----------+-----------------+-------------------------+-----+--------+----------+---------------+-----------+-----------+----------------+------------+
|personid                            |count|labcode|loincclass|interpretation   |servicedate              |value|modifier|refLowtype|refLowtextvalue|refLowRange|refHightype|refHightextvalue|refHighRange|
+------------------------------------+-----+-------+----------+-----------------+-------------------------+-----+--------+----------+---------------+-----------+-----------+----------------+------------+
|010f943d-aae3-42b4-b175-190914c45e7d|4    |31208-2|SPEC      |null             |2022-03-18T11:42:04+00:00|null |null    |null      |null           |null       |null       |null            |null        |
|010f943d-aae3-42b4-b175-190914c45e7d|4    |35592-5|CHEM      |null             |2020-11-22T19:42:07+00:00|64.06|null    |null      |null           |null       |null       |null       

<IPython.core.display.Javascript object>

In [32]:
# Filter the DataFrame by personid
filtered_df = Cohort_Lab_Original_allcols.filter(col("personid") == "1e6498a1-d6f9-40f5-8d8a-735f82d4dfc1")

# Show the filtered DataFrame
filtered_df.show(10, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------+----------+-----------------+-------------------------+-----+--------+----------+---------------+-----------+-----------+----------------+------------+
|personid                            |labcode|loincclass|interpretation   |servicedate              |value|modifier|refLowtype|refLowtextvalue|refLowRange|refHightype|refHightextvalue|refHighRange|
+------------------------------------+-------+----------+-----------------+-------------------------+-----+--------+----------+---------------+-----------+-----------+----------------+------------+
|1e6498a1-d6f9-40f5-8d8a-735f82d4dfc1|38445-3|CHEM      |Above high normal|2019-10-24T15:11:00+00:00|373.2|null    |null      |null           |null       |null       |null            |null        |
|1e6498a1-d6f9-40f5-8d8a-735f82d4dfc1|19146-0|MISC      |Not applicable   |2019-12-02T20:50:00+00:00|null |null    |null      |null           |null       |null       |null            |null        |
|1e6498a1-

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
Control_Lab_Original_allcols = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Control_Lab_Paired_Final_additionalColsFV.parquet")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
Epilepsy_Cohort_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Cohort_Lab_toStack_Cleaned")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
print(Epilepsy_Cohort_Lab.select("personid").distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

119609


<IPython.core.display.Javascript object>

In [4]:
print(Epilepsy_Cohort_Lab.select("labcode").distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

8836


<IPython.core.display.Javascript object>

In [3]:
Epilepsy_Control_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Control_Lab_toStack_Cleaned")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
# Filter the DataFrame by personid
filtered_df1 = Epilepsy_Cohort_Lab.filter(col("personid") == "1e6498a1-d6f9-40f5-8d8a-735f82d4dfc1")

# Show the filtered DataFrame
filtered_df1.show(10, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------+--------------------------+-----------+
|personid                            |labcode|New_updated_Interpretation|servicedate|
+------------------------------------+-------+--------------------------+-----------+
|1e6498a1-d6f9-40f5-8d8a-735f82d4dfc1|31208-2|Normal                    |2020-06-18 |
|1e6498a1-d6f9-40f5-8d8a-735f82d4dfc1|38445-3|High                      |2019-10-24 |
|1e6498a1-d6f9-40f5-8d8a-735f82d4dfc1|19146-0|null                      |2019-12-02 |
|1e6498a1-d6f9-40f5-8d8a-735f82d4dfc1|94500-6|Normal                    |2020-06-18 |
+------------------------------------+-------+--------------------------+-----------+



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
Epilepsy_Cohort_Lab.printSchema()
Epilepsy_Control_Lab.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- New_updated_Interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)

root
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- New_updated_Interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)



In [5]:
print(Epilepsy_Cohort_Lab.select("personid").distinct().count())
print(Epilepsy_Control_Lab.select("personid").distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

419825


<IPython.core.display.Javascript object>

In [23]:
print(Cohort_Lab_Original_allcols.select("personid").distinct().count())
print(Control_Lab_Original_allcols.select("personid").distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

419825


<IPython.core.display.Javascript object>

In [7]:
print(Epilepsy_Cohort_Lab.count())
print(Epilepsy_Control_Lab.count())

▸,:,


<IPython.core.display.Javascript object>

6273097


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

16273634


<IPython.core.display.Javascript object>

In [9]:
Epilepsy_Cohort_Lab_old1 = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/cohort_lab_Paired1")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
Epilepsy_Cohort_Lab_old1.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)



In [11]:
Epilepsy_Control_Lab_old1 = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/control_lab_Paired1.parquet")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
Epilepsy_Cohort_Lab_old1.printSchema()
Epilepsy_Control_Lab_old1.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)

root
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)



In [19]:
print(Epilepsy_Cohort_Lab_old1.select("personid").distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

119609


<IPython.core.display.Javascript object>

In [24]:
print(Epilepsy_Control_Lab_old1.select("personid").distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

419825


<IPython.core.display.Javascript object>

In [13]:
Epilepsy_Cohort_Lab_old2 = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/cohort_lab_Paired_Final.parquet")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
print(Epilepsy_Cohort_Lab_old2.select("personid").distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

119609


<IPython.core.display.Javascript object>

In [17]:
Epilepsy_Cohort_Lab_old2.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- loincclass: string (nullable = true)
 |-- interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)



In [34]:
from pyspark.sql.functions import col, count, when

# Calculate the total count of personid records for each labcode
labcode_cohort_counts = Epilepsy_Cohort_Lab.groupBy("labcode") \
    .agg(count("personid").alias("total_count"))

# Calculate the count of personid records where New_updated_Interpretation is null for each labcode
null_interpretation_cohort_counts = Epilepsy_Cohort_Lab.groupBy("labcode") \
    .agg(count(when(col("New_updated_Interpretation").isNull(), True)).alias("null_interpretation_count"))

# Join the two DataFrames to calculate the percentage of null interpretations
joined_cohort_counts = labcode_cohort_counts.join(null_interpretation_cohort_counts, "labcode", "inner")

# Calculate the percentage of null interpretations
joined_cohort_counts = joined_cohort_counts.withColumn("null_interpretation_percentage", 
                                         (col("null_interpretation_count") / col("total_count")) * 100)

# Filter labcodes where null interpretation percentage is greater than 90%
filtered_Cohort_labcodes = joined_cohort_counts.filter(col("null_interpretation_percentage") > 90)

# Count the number of such unique labcodes
count_filtered_Cohort_labcodes = filtered_Cohort_labcodes.select("labcode").distinct().count()

# Display the count of unique labcodes
print("Number of unique labcodes with more than 90% personid having null New_updated_Interpretation for cohort group:", count_filtered_Cohort_labcodes)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Number of unique labcodes with more than 90% personid having null New_updated_Interpretation for cohort group: 3515


<IPython.core.display.Javascript object>

In [35]:
filtered_Cohort_labcodes.printSchema()

▸,:,


root
 |-- labcode: string (nullable = true)
 |-- total_count: long (nullable = false)
 |-- null_interpretation_count: long (nullable = false)
 |-- null_interpretation_percentage: double (nullable = true)



In [36]:
filtered_Cohort_labcodes.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+---------+-----------+-------------------------+------------------------------+
|labcode  |total_count|null_interpretation_count|null_interpretation_percentage|
+---------+-----------+-------------------------+------------------------------+
|10706-0  |1          |1                        |100.0                         |
|121870001|77         |77                       |100.0                         |
|122296009|4          |4                        |100.0                         |
|16052-3  |38         |38                       |100.0                         |
|16276-8  |1          |1                        |100.0                         |
|18864-9  |15         |15                       |100.0                         |
|22682-9  |2          |2                        |100.0                         |
|24332-9  |25         |25                       |100.0                         |
|29161-7  |1          |1                        |100.0                         |
|30630007 |67         |67   

<IPython.core.display.Javascript object>

In [37]:
print(filtered_Cohort_labcodes.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

3515


<IPython.core.display.Javascript object>

In [38]:
print(filtered_Cohort_labcodes.select("labcode").distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

3515


<IPython.core.display.Javascript object>

In [39]:
from pyspark.sql.functions import col
# Perform left anti-join to remove records with labcode matching labcode from filtered_Cohort_labcodes
Epilepsy_Cohort_Lab_updated = Epilepsy_Cohort_Lab.join(filtered_Cohort_labcodes, "labcode", "left_anti")

# Show the resulting DataFrame
Epilepsy_Cohort_Lab_updated.show(truncate=False)
print(Epilepsy_Cohort_Lab_updated.count())
Epilepsy_Cohort_Lab_updated.printSchema()

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

6087453
root
 |-- labcode: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- New_updated_Interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)



<IPython.core.display.Javascript object>

In [40]:
print(Epilepsy_Cohort_Lab.count())

▸,:,


<IPython.core.display.Javascript object>

6273097


<IPython.core.display.Javascript object>

In [41]:
from pyspark.sql.functions import col, count, when

# Calculate the total count of personid records for each labcode
labcode_Control_counts1 = Epilepsy_Control_Lab.groupBy("labcode") \
    .agg(count("personid").alias("total_count"))

# Calculate the count of personid records where New_updated_Interpretation is null for each labcode
null_interpretation_Control_counts1 = Epilepsy_Control_Lab.groupBy("labcode") \
    .agg(count(when(col("New_updated_Interpretation").isNull(), True)).alias("null_interpretation_count"))

# Join the two DataFrames to calculate the percentage of null interpretations
joined_Control_counts1 = labcode_Control_counts1.join(null_interpretation_Control_counts1, "labcode", "inner")

# Calculate the percentage of null interpretations
joined_Control_counts1 = joined_Control_counts1.withColumn("null_interpretation_percentage", 
                                         (col("null_interpretation_count") / col("total_count")) * 100)

# Filter labcodes where null interpretation percentage is greater than 90%
filtered_Control_labcodes1 = joined_Control_counts1.filter(col("null_interpretation_percentage") > 90)

# Count the number of such unique labcodes
count_filtered_Control_labcodes1 = filtered_Control_labcodes1.select("labcode").distinct().count()

# Display the count of unique labcodes
print("Number of unique labcodes with more than 90% personid having null New_updated_Interpretation for cohort group:", count_filtered_Control_labcodes1)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Number of unique labcodes with more than 90% personid having null New_updated_Interpretation for cohort group: 4300


<IPython.core.display.Javascript object>

In [42]:
from pyspark.sql.functions import col
# Perform left anti-join to remove records with labcode matching labcode from filtered_Cohort_labcodes
Epilepsy_Control_Lab_updated = Epilepsy_Control_Lab.join(filtered_Control_labcodes1, "labcode", "left_anti")

# Show the resulting DataFrame
Epilepsy_Control_Lab_updated.show(truncate=False)
print(Epilepsy_Control_Lab_updated.count())
Epilepsy_Control_Lab_updated.printSchema()

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

15782777
root
 |-- labcode: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- New_updated_Interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)



<IPython.core.display.Javascript object>

In [43]:
print(Epilepsy_Control_Lab.count())

▸,:,


<IPython.core.display.Javascript object>

16273634


<IPython.core.display.Javascript object>

In [47]:
D_Epilepsy_Cohort_Lab_updated = Epilepsy_Cohort_Lab_updated.select("labcode").distinct()
D_Epilepsy_Control_Lab_updated = Epilepsy_Control_Lab_updated.select("labcode").distinct()
stacked_Lab = D_Epilepsy_Cohort_Lab_updated.union(D_Epilepsy_Control_Lab_updated)
D_stacked_Lab = stacked_Lab.select("labcode").distinct()
print(D_stacked_Lab.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

6589


<IPython.core.display.Javascript object>

In [48]:
D_Epilepsy_Cohort_Lab = Epilepsy_Cohort_Lab.select("labcode").distinct()
D_Epilepsy_Control_Lab = Epilepsy_Control_Lab.select("labcode").distinct()
stacked_Lab1 = D_Epilepsy_Cohort_Lab.union(D_Epilepsy_Control_Lab)
D1_stacked_Lab = stacked_Lab1.select("labcode").distinct()
print(D1_stacked_Lab.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

10825


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [49]:
Epilepsy_Cohort_Lab_updated.printSchema()
Epilepsy_Control_Lab_updated.printSchema()

▸,:,


root
 |-- labcode: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- New_updated_Interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)

root
 |-- labcode: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- New_updated_Interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)



In [51]:
##################################FinalCleaned for Stacking cohort-Lab############################################################
Epilepsy_Cohort_Lab_updated.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Cleaned_Lab_Cohort_toStack")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [52]:
##################################FinalCleaned for Stacking control-Lab############################################################
Epilepsy_Control_Lab_updated.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Cleaned_Lab_Control_toStack")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
Epilepsy_Cohort_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Cleaned_Lab_Cohort_toStack")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
print(Epilepsy_Cohort_Lab.select("personid").distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

118524


<IPython.core.display.Javascript object>

In [7]:
print(Epilepsy_Cohort_Lab.select("labcode").distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

5321


<IPython.core.display.Javascript object>